# E2 Phoneme-Level Steering (CTC-based)

Exploratory approach: train a simple CTC model to predict phonemes from encoder states,
then use phoneme-level representations for steering (inference-time friendly).

Compare with frame-level DTW steering to see if phoneme-level captures similar effects.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import jiwer
from collections import defaultdict

ROOT = Path("/rds/general/user/tsv22/home/accent-robust-asr")
sys.path.insert(0, str(ROOT))

from src.config import TEST_SPEAKERS, SPEAKER_L1
from src.training.evaluation.eval_whisfusion import WhisfusionWrapper
from src.utils.textgrid import parse_textgrid

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA A40


In [2]:
# Load phoneme inventory and helper functions
def load_checkpoint(path: str) -> np.ndarray:
    """Load encoder state from .pt file, return as float32 numpy array."""
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    hidden_states = checkpoint["hidden_states"]
    if isinstance(hidden_states, torch.Tensor):
        return hidden_states.to(torch.float32).cpu().numpy()
    return np.array(hidden_states, dtype=np.float32)

def get_phoneme_labels_from_textgrid(tg_path: str, sr: int = 16000, hop_length: int = 320):
    """Extract phoneme labels aligned to frames from TextGrid.
    
    Returns: list of phoneme labels, one per frame
    """
    segments = parse_textgrid(tg_path, tier_name='phones')
    
    # Create frame-wise labels (1500 frames total)
    n_frames = 1500
    frame_duration = hop_length / sr  # Duration of each frame in seconds
    frame_times = np.arange(n_frames) * frame_duration
    
    # Map each frame to its phoneme
    frame_phones = ['PAD'] * n_frames  # Default padding
    for segment in segments:
        start_frame = int(segment.start / frame_duration)
        end_frame = int(segment.end / frame_duration)
        phone = segment.text.upper()
        
        for i in range(max(0, start_frame), min(n_frames, end_frame + 1)):
            frame_phones[i] = phone
    
    return frame_phones

def get_phoneme_labels_from_lab(lab_path: str, sr: int = 16000, hop_length: int = 320):
    """Extract phoneme labels from CMU ARCTIC .lab file (same format as get_phoneme_labels_from_textgrid)."""
    segments = []
    with open(lab_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) >= 3:
                try:
                    start_time = float(parts[0])
                    end_time = float(parts[1]) if len(parts) >= 2 else start_time + 0.1
                    phone = parts[2].upper()
                    
                    # Store as segment object
                    class Segment:
                        def __init__(self, start, end, text):
                            self.start = start
                            self.end = end
                            self.text = text
                    
                    segments.append(Segment(start_time, end_time, phone))
                except (ValueError, IndexError):
                    continue
    
    # Convert to frame-wise labels
    n_frames = 1500
    frame_duration = hop_length / sr
    frame_phones = ['PAD'] * n_frames
    
    for segment in segments:
        start_frame = int(segment.start / frame_duration)
        end_frame = int(segment.end / frame_duration)
        
        for i in range(max(0, start_frame), min(n_frames, end_frame + 1)):
            frame_phones[i] = segment.text
    
    return frame_phones

print("✓ Helper functions defined")

✓ Helper functions defined


In [3]:
# Load mapping and data paths
mapping_cache = ROOT / "src/analysis/cache/utterance_mapping.json"
wer_csv = ROOT / "results/model_perf_comparison/whisfusion_finetuned_predictions.csv"

print(f"Loading utterance mapping...")
with open(mapping_cache) as f:
    mapping = json.load(f)

# Build phoneme inventory from all speakers
L2ARCTIC_ABS = ROOT / "data" / "l2_arctic"
CMU_ARCTIC_ABS = ROOT / "data" / "cmu_arctic"

# phoneme_set = set()
# for prompt_id, l1_dict in tqdm(mapping.items(), desc="Building phoneme inventory"):
#     for l1, info in l1_dict.items():
#         speaker = info["speaker"]
        
#         if l1 == "English":
#             lab_path = CMU_ARCTIC_ABS / f"cmu_us_{speaker.lower()}_arctic" / "lab" / f"{prompt_id}.lab"
#             if lab_path.exists():
#                 try:
#                     phones = get_phoneme_labels_from_lab(str(lab_path))
#                     phoneme_set.update(phones)
#                 except:
#                     pass
#         else:
#             tg_path = L2ARCTIC_ABS / speaker / "textgrid" / f"{prompt_id}.TextGrid"
#             if tg_path.exists():
#                 try:
#                     phones = get_phoneme_labels_from_textgrid(str(tg_path))
#                     phoneme_set.update(phones)
#                 except:
#                     pass

# phoneme_list = set(phoneme_set)
phoneme_list = ['AA', 'AE', 'AH', 'AO', 'AW', 'AX', 'AY', 'B', 'CH', 'D', 'DH', 'EH', 'ER', 'EY', 'F', 'G', 'HH', 'IH', 'IY', 'JH', 'K', 'L', 'M', 'N', 'NG', 'OW', 'OY', 'P', 'PAD', 'PAU', 'R', 'S', 'SH', 'T', 'TH', 'UH', 'UW', 'V', 'W', 'Y', 'Z', 'ZH']
phone2idx = {p: i for i, p in enumerate(phoneme_list)}
idx2phone = {i: p for p, i in phone2idx.items()}

print(f"✓ Loaded {len(phoneme_list)} phonemes: {phoneme_list}")

Loading utterance mapping...
✓ Loaded 42 phonemes: ['AA', 'AE', 'AH', 'AO', 'AW', 'AX', 'AY', 'B', 'CH', 'D', 'DH', 'EH', 'ER', 'EY', 'F', 'G', 'HH', 'IH', 'IY', 'JH', 'K', 'L', 'M', 'N', 'NG', 'OW', 'OY', 'P', 'PAD', 'PAU', 'R', 'S', 'SH', 'T', 'TH', 'UH', 'UW', 'V', 'W', 'Y', 'Z', 'ZH']


In [ ]:
# Simple CTC Dataset
class PhonemeDataset(Dataset):
    def __init__(self, mapping, split='train', phone2idx=None):
        self.mapping = mapping
        self.split = split
        self.phone2idx = phone2idx
        self.samples = []
        
        # Collect samples from specified split
        for prompt_id, l1_dict in mapping.items():
            for l1, speaker_info in l1_dict.items():
                speaker = speaker_info["speaker"]
                path = speaker_info["path"]
                
                # Filter by speaker split (train/test)
                is_test_speaker = speaker in TEST_SPEAKERS
                if split == 'train' and is_test_speaker:
                    continue
                if split == 'test' and not is_test_speaker:
                    continue
                
                # Get phoneme labels
                if l1 == "English":
                    lab_path = CMU_ARCTIC_ABS / f"cmu_us_{speaker.lower()}_arctic" / "lab" / f"{prompt_id}.lab"
                    if not lab_path.exists():
                        continue
                    try:
                        phones = get_phoneme_labels_from_lab(str(lab_path))
                    except:
                        continue
                else:
                    tg_path = L2ARCTIC_ABS / speaker / "textgrid" / f"{prompt_id}.TextGrid"
                    if not tg_path.exists():
                        continue
                    try:
                        phones = get_phoneme_labels_from_textgrid(str(tg_path))
                    except:
                        continue
                
                # Convert phone labels to indices
                phone_indices = [phone2idx.get(p, phone2idx.get('PAD', 0)) for p in phones]
                
                self.samples.append({
                    'enc_path': path,
                    'phone_indices': phone_indices,
                    'prompt_id': prompt_id,
                    'l1': l1,
                    'speaker': speaker
                })
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Load encoder state
        enc_state = load_checkpoint(sample['enc_path'])
        enc_state = torch.from_numpy(enc_state).float()
        
        # Phone indices
        phones = torch.tensor(sample['phone_indices'], dtype=torch.long)
        
        return {
            'encoder_state': enc_state,
            'phone_indices': phones,
            'prompt_id': sample['prompt_id'],
            'l1': sample['l1'],
            'speaker': sample['speaker']
        }

print("✓ Dataset class defined")

In [5]:
# Simple CTC Model
class SimpleCTCModel(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256, num_phonemes=50):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_phonemes)
    
    def forward(self, x):
        # x: (batch, seq_len, 768)
        lstm_out, _ = self.lstm(x)
        # lstm_out: (batch, seq_len, hidden_dim*2)
        logits = self.fc(lstm_out)
        # logits: (batch, seq_len, num_phonemes)
        return logits

print("✓ CTC model class defined")

✓ CTC model class defined


In [6]:
# Create train dataset (use all speakers for now in exploratory mode)
print("Creating dataset...")
train_dataset = PhonemeDataset(mapping, split='train', phone2idx=phone2idx)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Number of phonemes: {len(phoneme_list)}")

# Create dataloaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

print(f"Created train loader with {len(train_loader)} batches")

Creating dataset...
Train dataset size: 1132
Number of phonemes: 42
Created train loader with 71 batches


In [ ]:
# Train simple CTC model
model = SimpleCTCModel(input_dim=768, hidden_dim=256, num_phonemes=len(phoneme_list)).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
ctc_loss_fn = nn.CTCLoss(blank=phone2idx.get('PAD', 0), reduction='mean', zero_infinity=True)

num_epochs = 5
print(f"Training CTC model for {num_epochs} epochs...\n")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    
    for batch_idx, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")):
        encoder_states = batch['encoder_state'].to(device)  # (batch, 1500, 768)
        phone_indices = batch['phone_indices'].to(device)   # (batch, 1500)
        
        # Forward pass
        logits = model(encoder_states)  # (batch, 1500, num_phonemes)
        logits = logits.transpose(0, 1)  # CTCLoss expects (seq_len, batch, num_classes)
        
        # Input lengths (all are 1500)
        input_lengths = torch.full((encoder_states.size(0),), 1500, dtype=torch.long).to(device)
        target_lengths = torch.full((encoder_states.size(0),), 1500, dtype=torch.long).to(device)
        
        # Compute CTC loss
        loss = ctc_loss_fn(logits, phone_indices, input_lengths, target_lengths)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Avg Loss: {avg_loss:.4f}\n")

print("✓ Training complete")

Training CTC model for 5 epochs...



Epoch 1/5:   0%|          | 0/71 [00:00<?, ?it/s]

Epoch 1 - Avg Loss: 0.0000



Epoch 2/5:   0%|          | 0/71 [00:00<?, ?it/s]

Epoch 2 - Avg Loss: 0.0000



Epoch 3/5:   0%|          | 0/71 [00:00<?, ?it/s]

Epoch 3 - Avg Loss: 0.0000



Epoch 4/5:   0%|          | 0/71 [00:00<?, ?it/s]

In [ ]:
# Save trained model
model_path = ROOT / "src/analysis/results/e2_steering/ctc_model.pt"
model_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    'model_state': model.state_dict(),
    'phone2idx': phone2idx,
    'idx2phone': idx2phone,
    'phoneme_list': phoneme_list
}, model_path)

print(f"✓ Saved model to {model_path}")

In [ ]:
# Test CTC predictions on a sample
model.eval()

with torch.no_grad():
    sample = train_dataset[0]
    enc_state = sample['encoder_state'].unsqueeze(0).to(device)  # Add batch dimension
    logits = model(enc_state)  # (1, 1500, num_phonemes)
    
    # Get predicted phones (greedy decoding)
    pred_phones = torch.argmax(logits[0], dim=1)  # (1500,)
    pred_phone_labels = [idx2phone.get(int(p), 'UNK') for p in pred_phones]
    
    # Compare with ground truth
    true_phone_labels = [idx2phone.get(int(p), 'UNK') for p in sample['phone_indices']]
    
    # Calculate phone-level accuracy
    correct = sum(1 for t, p in zip(true_phone_labels, pred_phone_labels) if t == p)
    accuracy = correct / len(true_phone_labels)
    
    print(f"Sample: {sample['prompt_id']} ({sample['l1']})")
    print(f"Phone-level accuracy: {accuracy:.2%}")
    print(f"\nFirst 50 frames:")
    print(f"True:  {' '.join(true_phone_labels[:50])}")
    print(f"Pred:  {' '.join(pred_phone_labels[:50])}")

## Next steps (in development):

1. Use CTC predictions to get phoneme-level alignments
2. Mean-pool encoder states per predicted phoneme
3. Learn phoneme-level steering (e.g., which phonemes need steering)
4. Compare phoneme-level steering results with frame-level DTW steering
5. Test inference-time deployment (single utterance → phoneme sequence → steering)